In [6]:
from Bio import SeqIO
import pandas as pd

file_csv = "../../documentation/clean_entity_mapping.csv"
chain_type = "../../documentation/pdb_sequences.fasta"
def get_chain_df(file_csv, chain_type):
    df = pd.read_csv(file_csv)
    chain_column = "light_subclass"

    chains = {
        (str(row['pdb']).lower(), str(row[chain_column]).upper())
        for _, row in df.iterrows()
        if str(row[chain_column]).upper() not in ['NAN', '', 'NONE']
    }

    print(f"how many light chains are in the file: {len(chains)}")
    return chains


def extract_pdb_and_chains(header):
  
    parts = header.split("_")
    pdb_id = parts[0].lower()
    chain_part = next((p for p in parts if p.startswith("chain")), None)
    if chain_part:
        chain_str = chain_part[len("chain"):].split("_")[0]
        chains = [c.upper() for c in chain_str.split(",")]
        return pdb_id, chains
    return pdb_id, []


def extract_unique_chain_sequences(fasta_path, chain_mapping, output_path):
    count_total = 0
    unique_sequences = {}

    for record in SeqIO.parse(fasta_path, "fasta"):
        header = record.id
        sequence = str(record.seq)
        count_total += 1

        pdb_id, chain_ids = extract_pdb_and_chains(header)
        if not chain_ids:
            continue

        if any((pdb_id, chain) in chain_mapping for chain in chain_ids):
            if sequence not in unique_sequences:
                unique_sequences[sequence] = header

    with open(output_path, "w") as out_fasta:
        for seq, header in unique_sequences.items():
            out_fasta.write(f">{header}\n{seq}\n")

    print(f"from {count_total} ")
    print(f"{len(unique_sequences)} unique sequences")

OUTPUT_FASTA = f"{chain_type.lower()}_sequences_deduplicated.fasta"

chain_mapping = get_chain_df(file_csv, chain_type)
extract_unique_chain_sequences(file_csv, chain_mapping, OUTPUT_FASTA)


how many light chains are in the file: 1406
from 0 
0 unique sequences


/home/barth/miniforge3/lib/python3.12/site-packages/Bio/SeqIO/FastaIO.py:203: BiopythonDeprecationWarning: Previously, the FASTA parser silently ignored comments at the beginning of the FASTA file (before the first sequence).

Nowadays, the FASTA file format is usually understood not to have any such comments, and most software packages do not allow them. Therefore, the use of comments at the beginning of a FASTA file is now deprecated in Biopython.

In a future Biopython release, this deprecation warning will be replaced by a ValueError. To avoid this, there are three options:

(1) Modify your FASTA file to remove such comments at the beginning of the file.

(2) Use SeqIO.parse with the 'fasta-pearson' format instead of 'fasta'. This format is consistent with the FASTA format defined by William Pearson's FASTA aligner software. Thie format allows for comments before the first sequence; lines starting with the ';' character anywhere in the file are also regarded as comment lines and ar